In [1]:
# ==============================================================================
# SCRIPT 02: EMPIRICAL BENCHMARK EVALUATION & SOCIOECONOMIC SWEEP
# ==============================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("1. Loading Empirical USDA Catalog from Dropbox...")
url = "https://www.dropbox.com/scl/fi/576w2ca4qhqm57vug2ae0/usda_empirical_catalog.csv?rlkey=lor65embs1yrbl9sghzlv8jzv&dl=1"
df = pd.read_csv(url)
print(f"-> Successfully loaded {len(df)} empirical records.")

if 'AddedSugar_g' not in df.columns:
    sugar_cols = [c for c in df.columns if 'sugar' in c.lower()]
    if sugar_cols:
        df = df.rename(columns={sugar_cols[0]: 'AddedSugar_g'})
        print(f"-> Note: Automatically mapped '{sugar_cols[0]}' to 'AddedSugar_g' to fix KeyError.")
    else:
        raise KeyError("Could not find any column containing the word 'Sugar' in the dataset.")
# ---------------------------------

# 1. BASELINE PARAMETERS
W_Price, W_Time = 0.50, 0.50
tau_sodium = 600.0
tau_satfat = 3.0
tau_sugar = 5.0
lambda_penalty = 0.10
J_candidates = 50

# 2. UTILITY NORMALIZATION (Catalog-dependent)
P_max, P_min = df['Price_USD'].max(), df['Price_USD'].min()
T_max, T_min = df['Prep_Time_min'].max(), df['Prep_Time_min'].min()
P_denom = (P_max - P_min) if (P_max - P_min) > 0 else 1
T_denom = (T_max - T_min) if (T_max - T_min) > 0 else 1

df['U_Price'] = (P_max - df['Price_USD']) / P_denom
df['U_Time'] = (T_max - df['Prep_Time_min']) / T_denom
df['Utility'] = (W_Price * df['U_Price']) + (W_Time * df['U_Time'])

# 3. SOCIOECONOMIC SWEEP (Varying Budget Beta)
print("\n--- SOCIOECONOMIC HETEROGENEITY ANALYSIS (VARYING BUDGET) ---")
budgets_to_test = [0.40, 0.60, 0.80, 1.20]
results_beta = []

for beta_budget in budgets_to_test:

    # Violations based on current beta
    df['Viol_Budget'] = np.maximum(0, df['Price_USD'] - beta_budget) / beta_budget
    df['Viol_Sodium'] = np.maximum(0, df['Sodium_mg'] - tau_sodium) / tau_sodium
    df['Viol_SatFat'] = np.maximum(0, df['SatFat_g'] - tau_satfat) / tau_satfat
    df['Viol_Sugar'] = np.maximum(0, df['AddedSugar_g'] - tau_sugar) / tau_sugar

    # M0
    best_m0 = df.loc[df['Utility'].idxmax()]

    # M1
    top_J = df.nlargest(J_candidates, 'Utility')
    valid_m1 = top_J[(top_J['Sodium_mg'] <= tau_sodium) & (top_J['SatFat_g'] <= tau_satfat) & (top_J['AddedSugar_g'] <= tau_sugar) & (top_J['Price_USD'] <= beta_budget)]
    best_m1 = valid_m1.iloc[0] if len(valid_m1) > 0 else None

    # M2
    df['U_Soft'] = df['Utility'] - lambda_penalty * (df['Viol_Budget'] + df['Viol_Sodium'] + df['Viol_SatFat'] + df['Viol_Sugar'])
    best_m2 = df.loc[df['U_Soft'].idxmax()]

    # M3 (Pure Two-Step Lexicographic)
    strict_mask = (df['Price_USD'] <= beta_budget) & (df['Sodium_mg'] <= tau_sodium) & (df['SatFat_g'] <= tau_satfat) & (df['AddedSugar_g'] <= tau_sugar)
    if strict_mask.any():
        best_m3 = df[strict_mask].loc[df[strict_mask]['Utility'].idxmax()]
    else:
        df['Z_Deviation'] = df['Viol_Budget'] + df['Viol_Sodium'] + df['Viol_SatFat'] + df['Viol_Sugar']
        min_Z = df['Z_Deviation'].min()
        # Tolerance added to account for floating point errors in Z_Deviation equality
        min_Z_items = df[np.isclose(df['Z_Deviation'], min_Z, atol=1e-8)]
        best_m3 = min_Z_items.loc[min_Z_items['Utility'].idxmax()]

    m1_res = "Empty Set" if best_m1 is None else f"${best_m1['Price_USD']:.2f}"

    results_beta.append({
        "Budget": f"${beta_budget:.2f}",
        "M1_Result": m1_res,
        "M2_Price": f"${best_m2['Price_USD']:.2f}",
        "M3_Price": f"${best_m3['Price_USD']:.2f}"
    })

df_beta_sweep = pd.DataFrame(results_beta)
print(df_beta_sweep.to_string(index=False))

# 4. GENERATE LATEX TABLE 2 (USING BASELINE BETA = 0.60)
print("\n--- GENERATING LATEX TABLE 2 (BETA = $0.60) ---")
# Re-run for standard Beta=0.60 to populate the table
beta_budget = 0.60
df['Viol_Budget'] = np.maximum(0, df['Price_USD'] - beta_budget) / beta_budget
df['U_Soft'] = df['Utility'] - lambda_penalty * (df['Viol_Budget'] + df['Viol_Sodium'] + df['Viol_SatFat'] + df['Viol_Sugar'])
best_m0 = df.loc[df['Utility'].idxmax()]
best_m1 = None # Known empty set at 0.60
best_m2 = df.loc[df['U_Soft'].idxmax()]
df['Z_Deviation'] = df['Viol_Budget'] + df['Viol_Sodium'] + df['Viol_SatFat'] + df['Viol_Sugar']
min_Z = df['Z_Deviation'].min()
min_Z_items = df[np.isclose(df['Z_Deviation'], min_Z, atol=1e-8)]
best_m3 = min_Z_items.loc[min_Z_items['Utility'].idxmax()]

models = [
    ("M0: Baseline", best_m0),
    ("M1: Post-Hoc", best_m1),
    ("M2: Soft Penalty", best_m2),
    ("M3: Goal Programming", best_m3)
]

latex_t2 = "\\begin{table}[h!]\n\\centering\n"
latex_t2 += "\\caption{Empirical Benchmark Results (USDA PP-NAP \\& FNDDS)}\n"
latex_t2 += "\\label{tab:empirical_results}\n"
latex_t2 += "\\resizebox{\\textwidth}{!}{%\n"
latex_t2 += "\\begin{tabular}{llccccc}\n\\toprule\n"
latex_t2 += "\\textbf{Model} & \\textbf{Product} & \\textbf{Price (\\$)} & \\textbf{Time (min)} & \\textbf{Sodium (mg)} & \\textbf{Sat. Fat (g)} & \\textbf{Total Sugar (g)} \\\\\n\\midrule\n"

for name, p in models:
    if p is None:
        latex_t2 += f"{name} & \\textit{{Infeasible (Empty Candidate Set)}} & -- & -- & -- & -- & -- \\\\\n"
    else:
        prod_name = str(p['Product_Name']).title()[:35].replace('&', '\\&')
        price, time = f"{p['Price_USD']:.2f}", f"{p['Prep_Time_min']:.1f}"
        sod, fat, sug = f"{p['Sodium_mg']:.0f}", f"{p['SatFat_g']:.1f}", f"{p['AddedSugar_g']:.1f}"

        if p['Sodium_mg'] > tau_sodium: sod = f"\\textbf{{{sod}}}"
        if p['SatFat_g'] > tau_satfat: fat = f"\\textbf{{{fat}}}"
        if p['AddedSugar_g'] > tau_sugar: sug = f"\\textbf{{{sug}}}"

        latex_t2 += f"{name} & {prod_name} & {price} & {time} & {sod} & {fat} & {sug} \\\\\n"

latex_t2 += "\\bottomrule\n"
latex_t2 += "\\multicolumn{7}{p{15cm}}{\\footnotesize \\textit{Note:} Illustrative item-level thresholds ($\\tau_{sod}=600$mg, $\\tau_{fat}=3.0$g, $\\tau_{sug}=5.0$g) exceeded are marked in bold. $\\beta = \\$0.60$, $\\lambda = 0.1$.}\n"
latex_t2 += "\\end{tabular}%\n}\n\\end{table}\n"
print(latex_t2)

1. Loading Empirical USDA Catalog from Dropbox...
-> Successfully loaded 385 empirical records.
-> Note: Automatically mapped 'TotalSugar_g' to 'AddedSugar_g' to fix KeyError.

--- SOCIOECONOMIC HETEROGENEITY ANALYSIS (VARYING BUDGET) ---
Budget M1_Result M2_Price M3_Price
 $0.40 Empty Set    $0.13    $0.28
 $0.60 Empty Set    $0.13    $0.28
 $0.80     $0.67    $0.13    $0.67
 $1.20     $0.67    $0.13    $0.67

--- GENERATING LATEX TABLE 2 (BETA = $0.60) ---
\begin{table}[h!]
\centering
\caption{Empirical Benchmark Results (USDA PP-NAP \& FNDDS)}
\label{tab:empirical_results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llccccc}
\toprule
\textbf{Model} & \textbf{Product} & \textbf{Price (\$)} & \textbf{Time (min)} & \textbf{Sodium (mg)} & \textbf{Sat. Fat (g)} & \textbf{Total Sugar (g)} \\
\midrule
M0: Baseline & Soup, Ramen Noodles, Water Added & 0.13 & 3.0 & \textbf{705} & \textbf{3.0} & 0.8 \\
M1: Post-Hoc & \textit{Infeasible (Empty Candidate Set)} & -- & -- & -- & -- & -- \\
M2: So